In [1]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.12.2 environment at: /Users/ethanbobrik/Projects/RAG LLMOps/.venv
Audited 6 packages in 51ms


In [2]:
from dotenv import load_dotenv

import os

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


# Data Ingestion

In [3]:
from langchain_community.document_loaders import TextLoader

/var/folders/jz/hkhgc4gx2dj_k8h_29_96nlw0000gn/T/ipykernel_27604/2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
loader = TextLoader("../data/Agentic AI.txt", encoding='utf8')
docs = loader.load()

In [5]:
docs[0].page_content[:500]

"Agentic AI\n==========\n\nOverview\n--------\nAgentic AI refers to artificial intelligence systems that can act autonomously to\npursue goals on a user's behalf. Unlike a traditional model that produces a single\nresponse to a single prompt, an agentic system can plan a sequence of steps, take\nactions in the world (calling tools, APIs, or other software), observe the results,\nand adjust its approach until the goal is met. The defining quality is agency: the\nsystem decides what to do next rather than wa"

In [6]:
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

In [8]:
text_chunks = text_splitter.split_documents(docs)

In [9]:
text_chunks

[Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='Agentic AI\n=========='),
 Document(metadata={'source': '../data/Agentic AI.txt'}, page_content="Overview\n--------\nAgentic AI refers to artificial intelligence systems that can act autonomously to\npursue goals on a user's behalf. Unlike a traditional model that produces a single"),
 Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='response to a single prompt, an agentic system can plan a sequence of steps, take\nactions in the world (calling tools, APIs, or other software), observe the results,'),
 Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='and adjust its approach until the goal is met. The defining quality is agency: the\nsystem decides what to do next rather than waiting for a human to spell out every step.'),
 Document(metadata={'source': '../data/Agentic AI.txt'}, page_content='Core Characteristics\n--------------------\n1. Goal-directed behavior. An agent is gi

In [10]:
! uv pip install faiss-cpu langchain_openai

Using Python 3.12.2 environment at: /Users/ethanbobrik/Projects/RAG LLMOps/.venv
Audited 2 packages in 3ms


In [11]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_classic.vectorstores import FAISS

In [12]:
embeddings = OpenAIEmbeddings()

In [13]:
vectorstore = FAISS.from_documents(text_chunks, embeddings)

In [14]:
vectorstore

In [15]:
retriever = vectorstore.as_retriever()

In [16]:
# perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-"*50)

Document 1:
Agentic AI
--------------------------------------------------
Document 2:
Overview
--------
Agentic AI refers to artificial intelligence systems that can act autonomously to
pursue goals on a user's behalf. Unlike a traditional model that produces a single
--------------------------------------------------
Document 3:
The Road Ahead
--------------
Agentic AI is moving from demos toward production. Progress depends on better planning
--------------------------------------------------
Document 4:
How Agentic Systems Work
------------------------
Most modern agents are built around a large language model (LLM) acting as the
"reasoning engine." A typical loop looks like this:
--------------------------------------------------


In [17]:
from langchain_classic.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [18]:
prompt = ChatPromptTemplate.from_template(template)

In [19]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [20]:
from langchain_classic.schema.output_parser import StrOutputParser

In [21]:
output_parser = StrOutputParser()

In [22]:
from langchain_openai.chat_models import ChatOpenAI

llm_model = ChatOpenAI(model_name="gpt-4o-mini")

In [23]:
from langchain_classic.schema.runnable import RunnablePassthrough

rag_chain = (
    {"context":retriever, "question": RunnablePassthrough()} | prompt | llm_model | output_parser
)

In [24]:
rag_chain.invoke("tell me about Agentic AI")

"Agentic AI refers to artificial intelligence systems designed to act autonomously on a user's behalf to pursue specific goals. Unlike traditional AI models that typically generate a single output, Agentic AI systems can independently initiate actions and make decisions. These systems often utilize large language models (LLMs) as their reasoning engines, enabling them to interpret and respond to complex tasks. Currently, the development of Agentic AI is transitioning from demonstration phases into practical applications. Progress in this field largely relies on improved planning and execution strategies. Overall, Agentic AI represents a significant evolution in how AI can assist users by enhancing autonomy and adaptability in decision-making processes."